In [1]:
pip install pandas faker sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.1 MB/s eta 0:00:00


In [2]:
import time
import json
import random
import hashlib
import sqlite3
import logging
import tempfile
from datetime import datetime, timedelta
from typing import Any, Callable, Dict, List, Optional
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path

# ── Bağımlılıklar (pip install pandas faker sqlalchemy) ──────────────────────
import pandas as pd
from faker import Faker

# ── Loglama Yapılandırması ───────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("airflow.simulation")

fake = Faker("tr_TR")  # Türkçe sahte veri üreteci


DEFAULT_DB_PATH = str(Path(tempfile.gettempdir()) / "satis_warehouse.db")


# =============================================================================
# ❶  AIRFLOW CORE — Temel Sınıflar (Simülasyon)
# =============================================================================

class TaskState(Enum):
    """Task'ın olası durumları (Airflow'da aynı enum mevcuttur)."""
    PENDING   = "pending"    # Henüz çalışmadı
    RUNNING   = "running"    # Şu an çalışıyor
    SUCCESS   = "success"    # Başarıyla tamamlandı
    FAILED    = "failed"     # Hata ile sonuçlandı
    SKIPPED   = "skipped"    # Atlandı


@dataclass
class XComStore:
    """
    XCom (Cross-Communication) — Task'lar arası mesaj kutusu.
    Gerçek Airflow'da bu veriler metadata DB'de saklanır.
    Küçük veri paylaşımı için tasarlanmıştır (büyük veri için S3/GCS kullanın).
    """
    _store: Dict[str, Any] = field(default_factory=dict)

    def push(self, task_id: str, key: str, value: Any):
        """Bir task'ın ürettiği veriyi depola."""
        store_key = f"{task_id}::{key}"
        self._store[store_key] = value
        log.debug(f"XCom PUSH → {store_key}")

    def pull(self, task_id: str, key: str) -> Any:
        """Bir task'ın daha önce depoladığı veriyi çek."""
        store_key = f"{task_id}::{key}"
        value = self._store.get(store_key)
        log.debug(f"XCom PULL ← {store_key} = {type(value).__name__}")
        return value


@dataclass
class TaskInstance:
    """
    Airflow TaskInstance: Bir task'ın tek bir çalışma kaydı.
    task_id, dag_id, execution_date bilgilerini tutar.
    """
    task_id: str
    dag_id: str
    execution_date: datetime
    xcom: XComStore
    state: TaskState = TaskState.PENDING
    start_date: Optional[datetime] = None
    end_date: Optional[datetime] = None
    duration: float = 0.0
    log_messages: List[str] = field(default_factory=list)

    def xcom_push(self, key: str, value: Any):
        """Bu task'tan veri yayınla."""
        self.xcom.push(self.task_id, key, value)

    def xcom_pull(self, task_ids: str, key: str) -> Any:
        """Başka bir task'ın verisini çek."""
        return self.xcom.pull(task_ids, key)

    def log_info(self, msg: str):
        self.log_messages.append(msg)
        log.info(f"[{self.task_id}] {msg}")


class PythonOperator:
    """
    Airflow PythonOperator: Python fonksiyonu çalıştıran temel operatör.
    Gerçek kullanım:
        task = PythonOperator(
            task_id='my_task',
            python_callable=my_function,
            op_kwargs={'param': 'value'},
            dag=dag,
        )
    """
    def __init__(
        self,
        task_id: str,
        python_callable: Callable,
        op_kwargs: Optional[Dict] = None,
        retries: int = 1,
        retry_delay: int = 2,  # saniye
        group: str = "default",
        on_failure_callback: Optional[Callable] = None, #mail için eklendi
    ):
        self.task_id = task_id
        self.python_callable = python_callable
        self.op_kwargs = op_kwargs or {}
        self.retries = retries
        self.retry_delay = retry_delay
        self.group = group
        self.upstream_tasks: List[str] = []    # Bu task'tan önce çalışacaklar
        self.downstream_tasks: List[str] = []  # Bu task'tan sonra çalışacaklar
        self.on_failure_callback = on_failure_callback #mail için eklendi

    def set_upstream(self, task: "PythonOperator"):
        """Bağımlılık tanımla: self, task'tan SONRA çalışır."""
        if task.task_id not in self.upstream_tasks:
            self.upstream_tasks.append(task.task_id)
        if self.task_id not in task.downstream_tasks:
            task.downstream_tasks.append(self.task_id)

    def __rshift__(self, other: "PythonOperator"):
        """>> operatörü: task1 >> task2 şeklinde bağımlılık tanımlar."""
        other.set_upstream(self)
        return other


class DAG:
    """
    Airflow DAG (Directed Acyclic Graph): Tüm iş akışının konteyneri.

    Gerçek Airflow'da DAG dosyası $AIRFLOW_HOME/dags/ içine konur ve
    scheduler periyodik olarak tarar.

    Örnek schedule değerleri:
        "@daily"      → Her gün gece yarısı
        "@hourly"     → Her saat başı
        "0 6 * * *"   → Her gün 06:00'da (cron syntax)
        None          → Manuel tetikleme
    """
    def __init__(
        self,
        dag_id: str,
        description: str,
        schedule_interval: Optional[str],
        start_date: datetime,
        catchup: bool = False,
        tags: Optional[List[str]] = None,
        default_args: Optional[Dict] = None,
    ):
        self.dag_id = dag_id
        self.description = description
        self.schedule_interval = schedule_interval
        self.start_date = start_date
        self.catchup = catchup
        self.tags = tags or []
        self.default_args = default_args or {}
        self.tasks: Dict[str, PythonOperator] = {}

    def add_task(self, task: PythonOperator):
        """DAG'a görev ekle."""
        self.tasks[task.task_id] = task

    def run(self, execution_date: Optional[datetime] = None) -> Dict[str, Any]:
        """
        DAG'ı çalıştır — topolojik sıralama ile görevleri işle.
        Gerçek Airflow'da bu işi Executor (LocalExecutor, CeleryExecutor) yapar.
        """
        execution_date = execution_date or datetime.now()
        xcom = XComStore()

        print("\n" + "═" * 70)
        print(f"  🚀  DAG ÇALIŞIYOR: {self.dag_id}")
        print(f"  📅  Execution Date: {execution_date.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"  📋  Açıklama: {self.description}")
        print("═" * 70)

        # Topolojik sıralama (bağımlılık sırasına göre çalıştır)
        ordered = self._topological_sort()
        results: Dict[str, TaskInstance] = {}
        dag_start = time.time()

        for task_id in ordered:
            task = self.tasks[task_id]
            ti = TaskInstance(
                task_id=task_id,
                dag_id=self.dag_id,
                execution_date=execution_date,
                xcom=xcom,
            )

            # Upstream task'ların başarılı olup olmadığını kontrol et
            failed_upstream = [
                uid for uid in task.upstream_tasks
                if results.get(uid) and results[uid].state == TaskState.FAILED
            ]
            if failed_upstream:
                ti.state = TaskState.SKIPPED
                ti.log_info(f"Atlandı — başarısız upstream: {failed_upstream}")
                results[task_id] = ti
                continue

            # Task'ı retry mantığıyla çalıştır
            self._execute_task(task, ti)
            results[task_id] = ti

        # Özet raporu
        dag_duration = time.time() - dag_start
        self._print_summary(results, dag_duration)
        return results

    def _execute_task(self, task: PythonOperator, ti: TaskInstance):
        """Tek bir task'ı yeniden deneme (retry) desteğiyle çalıştır."""
        group_icon = {"extract": "📥", "transform": "⚙️",
                      "load": "💾", "validate": "✅", "notify": "📣"
                      }.get(task.group, "🔧")

        print(f"\n  {group_icon}  Task: {ti.task_id}  [{task.group.upper()}]")
        print(f"      {'─' * 50}")

        for attempt in range(task.retries + 1):
            try:
                ti.state = TaskState.RUNNING
                ti.start_date = datetime.now()

                # Asıl fonksiyonu çalıştır — ti (TaskInstance) inject et
                task.python_callable(ti=ti, **task.op_kwargs)

                ti.end_date = datetime.now()
                ti.duration = (ti.end_date - ti.start_date).total_seconds()
                ti.state = TaskState.SUCCESS
                print(f"      ✓  Süre: {ti.duration:.2f}s")
                return

            except Exception as exc:
                if attempt < task.retries:
                    ti.log_info(f"Hata (deneme {attempt+1}/{task.retries}): {exc}")
                    print(f"      ⚠  Yeniden deneniyor ({attempt+1}/{task.retries})...")
                    time.sleep(task.retry_delay)
                else:
                    ti.end_date = datetime.now()
                    ti.duration = (ti.end_date - ti.start_date).total_seconds()
                    ti.state = TaskState.FAILED
                    if task.on_failure_callback:
                       task.on_failure_callback(ti, exc)
                    ti.log_info(f"BAŞARISIZ: {exc}")
                    print(f"      ✗  HATA: {exc}")

    def notify_failure(ti: TaskInstance, error: Exception):

        print("\n" + "═" * 70)
        print("📣 HATA BİLDİRİMİ")
        print(f"❌ '{ti.task_id}' görevi başarısız oldu.")
        print(f"🕒 Zaman : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"⚠️ Hata  : {error}")
        print("═" * 70)

    def _topological_sort(self) -> List[str]:
        """Kahn algoritması ile topolojik sıralama."""
        in_degree = {tid: 0 for tid in self.tasks}
        for task in self.tasks.values():
            for downstream in task.downstream_tasks:
                if downstream in in_degree:
                    in_degree[downstream] += 1

        queue = [tid for tid, deg in in_degree.items() if deg == 0]
        ordered = []

        while queue:
            tid = queue.pop(0)
            ordered.append(tid)
            for downstream in self.tasks[tid].downstream_tasks:
                if downstream in in_degree:
                    in_degree[downstream] -= 1
                    if in_degree[downstream] == 0:
                        queue.append(downstream)

        return ordered

    def _print_summary(self, results: Dict[str, TaskInstance], duration: float):
        """Pipeline sonuç özetini yazdır."""
        success = sum(1 for ti in results.values() if ti.state == TaskState.SUCCESS)
        failed  = sum(1 for ti in results.values() if ti.state == TaskState.FAILED)
        skipped = sum(1 for ti in results.values() if ti.state == TaskState.SKIPPED)

        print("\n" + "═" * 70)
        print(f"  📊  DAG TAMAMLANDI: {self.dag_id}")
        print(f"  ⏱   Toplam süre  : {duration:.2f} saniye")
        print(f"  ✅  Başarılı     : {success} task")
        print(f"  ❌  Başarısız    : {failed} task")
        print(f"  ⏭   Atlandı      : {skipped} task")
        print("═" * 70 + "\n")


# =============================================================================
# ❷  EXTRACT — Veri Çekme Görevleri
# =============================================================================

import pandas as pd
import random
from datetime import datetime
# Removed: from airflow.models import TaskInstance

# =============================================================================
# ❷  EXTRACT — Veri Çekme Görevleri (Gelişmiş Senaryo)
# =============================================================================

def extract_logistics_data(ti: TaskInstance, source_system: str = "SAP_ERP"):
    """
    [EXTRACT TASK] Global lojistik sisteminden sevkiyat verilerini çek.
    Senaryo: Farklı ülkelerden gelen, para birimleri karışık ve
    bazı alanları eksik (kirli) veriyi simüle eder.
    """
    # Note: ti.get_dagrun() and ti.log are attributes of a real Airflow TaskInstance.
    # In this simulation, TaskInstance does not have these attributes.
    # We'll adjust to use the simulated logging and a placeholder for execution_date.
    execution_date = ti.execution_date.strftime("%Y-%m-%d %H:%M")
    ti.log_info(f"--- {source_system} sisteminden veri çekme işlemi başladı ---")
    ti.log_info(f"Veri çekme periyodu: {execution_date}")

    # 1. Veri Simülasyonu (Gerçekçi Kirli Veri Seti)
    records = []
    countries = ["TR", "US", "DE", "CN", "UK"]
    carriers = ["DHL", "FedEx", "UPS", "Aras", "Ptt"]

    for i in range(200):
        # Bilinçli olarak bazı kayıtları "bozuk" oluşturuyoruz (Transform testi için)
        is_corrupted = random.random() < 0.1  # %10 ihtimalle kirli veri

        records.append({
            "tracking_id"  : f"TRK-{random.randint(100000, 999999)}",
            "order_id"     : fake.uuid4() if not is_corrupted else None, # Null ID hatası
            "origin"       : random.choice(countries),
            "destination"  : random.choice(countries),
            "carrier"      : random.choice(carriers),
            "weight_kg"    : round(random.uniform(0.5, 500.0), 2),
            "shipping_cost": round(random.uniform(5.0, 150.0), 2),
            "currency"     : "USD" if random.random() > 0.2 else "EUR", # Karışık para birimi
            "status"       : random.choice(["In Transit", "Delivered", "Returned", "Lost", "pending"]),
            "extracted_at" : datetime.now().isoformat()
        })

    df = pd.DataFrame(records)

    # 2. Veri Kaynağı Sağlık Kontrolü (Data Quality Check 1)
    if df['tracking_id'].duplicated().any():
        ti.log_info("Dikkat: Çekilen veride mükerrer tracking_id tespit edildi!") # Changed from ti.log.warning

    # 3. Metadata ekle ve XCom'a gönder
    # Veriyi JSON'a çevirirken 'iso' date formatını koruyoruz
    ti.xcom_push(key="raw_logistics_data", value=df.to_json(orient="records"))
    ti.xcom_push(key="source_info", value={"system": source_system, "row_count": len(df)})

    ti.log_info(f"İşlem Tamamlandı: {len(df)} adet sevkiyat kaydı XCom'a basıldı.")


def extract_warehouse_stock(ti: TaskInstance):
    """
    [EXTRACT TASK] Depo stok yönetim sisteminden (WMS) anlık stok verisi çek.
    Senaryo: Kritik stok seviyesi kontrolü ve hata yönetimi içerir.
    """
    ti.log_info("Depo (WMS) API bağlantısı kuruluyor...")

    # API'den veri gelmediği veya boş geldiği durumu simüle edelim
    # Şans eseri hata fırlatmak için:
    if random.random() < 0.05: # %5 ihtimalle sistem hatası simülasyonu
        ti.log_info("Kritik Hata: WMS API yanıt vermiyor (Timeout)!") # Changed from ti.log.error
        raise Exception("Depo yönetim sistemine erişilemedi. Pipeline durduruluyor.")

    # Gerçekçi stok verisi
    stock_data = [
        {
            "sku"          : f"SKU-{random.randint(1000, 2000)}",
            "warehouse_id" : random.choice(["IST-01", "ANK-02", "IZM-03"]),
            "stock_level"  : random.randint(0, 1000),
            "reorder_point": 50,
            "last_updated" : datetime.now().isoformat()
        }
        for _ in range(50)
    ]

    df_stock = pd.DataFrame(stock_data)

    # İş Mantığı Kontrolü: Eğer tüm stoklar 0 ise bir terslik vardır
    if df_stock["stock_level"].sum() == 0:
        ti.log_info("Veri Tutarsızlığı: Tüm stok seviyeleri 0 görünüyor. Kaynak veriyi kontrol edin!") # Changed from ti.log.error
        raise ValueError("Anormal veri: Boş stok verisi çekildi.")

    # XCom Push
    ti.xcom_push(key="inventory_data", value=df_stock.to_json(orient="records"))
    ti.log_info(f"Stok verisi başarıyla çekildi. Toplam SKU sayısı: {len(df_stock)}")

    # Transform aşamasına ipucu gönder (Context Injection)
    ti.xcom_push(key="low_stock_threshold", value=20)

# =============================================================================
# ❸  TRANSFORM — Veri Dönüşüm Görevleri
# =============================================================================

def clean_and_validate_data(ti: TaskInstance):
    """
    [TRANSFORM TASK] Ham veriyi temizle ve standartlaştır.

    İşlemler:
        1. Null değerleri ele al
        2. Geçersiz aralıkları düzelt/filtrele
        3. Metin normalizasyonu (büyük/küçük harf)
        4. Veri tipi dönüşümleri
        5. Duplicate kayıtları kaldır
    """
    ti.log_info("Veri temizleme başlıyor...")

    # XCom'dan önceki task'ın verisini çek
    # raw_json = ti.xcom_pull("extract_sales_data", "raw_data") # This task is not defined yet, using raw_logistics_data from extract_logistics_data for now.
    raw_json = ti.xcom_pull("extract_logistics_data", "raw_logistics_data")
    df = pd.read_json(pd.io.common.StringIO(raw_json), orient="records")
    initial_count = len(df)

    # 1. Status normalizasyonu (büyük harf → küçük harf)
    df["status"] = df["status"].str.lower().str.strip()

    # 2. Geçersiz status değerlerini filtrele
    valid_statuses = {"in transit", "delivered", "returned", "lost", "pending"}
    df = df[df["status"].isin(valid_statuses)]

    # 3. İskonto değeri düzeltme (0-100 arası olmalı) - This is for sales data, not logistics. Removing or adapting.
    # df["discount_pct"] = df["discount_pct"].fillna(0)
    # df["discount_pct"] = df["discount_pct"].clip(0, 100)
    # For logistics, we might want to check weight_kg or shipping_cost for validity
    df["weight_kg"] = df["weight_kg"].clip(0.1, 1000.0) # Ensure positive and reasonable weight
    df["shipping_cost"] = df["shipping_cost"].clip(0.0, 10000.0) # Ensure positive shipping cost

    # 4. Eksik kritik alanları kaldır
    # df = df.dropna(subset=["order_id", "customer_id", "unit_price"]) # For sales data
    df = df.dropna(subset=["tracking_id", "order_id", "origin", "destination"]) # For logistics data

    # 5. Duplicate order_id'leri kaldır (son kaydı tut)
    # df = df.drop_duplicates(subset=["order_id"], keep="last") # For sales data
    df = df.drop_duplicates(subset=["tracking_id"], keep="last") # For logistics data

    # 6. Tarih tipini düzelt - Assuming 'extracted_at' as the relevant datetime column
    # df["order_date"] = pd.to_datetime(df["order_date"])
    df["extracted_at"] = pd.to_datetime(df["extracted_at"])

    removed = initial_count - len(df)
    ti.log_info(f"Temizleme tamamlandı: {removed} kayıt kaldırıldı, {len(df)} kaldı.")

    ti.xcom_push("clean_data", df.to_json(orient="records", date_format="iso"))
    ti.xcom_push("removed_count", removed)


def enrich_and_aggregate(ti: TaskInstance):
    """
    [TRANSFORM TASK] Veriyi zenginleştir ve iş metriklerini hesapla.

    İşlemler:
        1. Toplam tutar hesaplama
        2. Müşteri segmenti ile join
        3. Bölge bazlı özet (aggregation)
        4. Kategori bazlı KPI'lar
    """
    ti.log_info("Veri zenginleştirme ve agregasyon başlıyor...")

    # Temiz satış verisini çek
    clean_json    = ti.xcom_pull("clean_and_validate_data", "clean_data")
    # customer_json = ti.xcom_pull("extract_customer_data", "customer_data") # This task is not defined in the current simulation
    # We'll use a dummy customer_data or skip customer join for now.
    df_logistics    = pd.read_json(pd.io.common.StringIO(clean_json), orient="records")
    # df_customers = pd.read_json(pd.io.common.StringIO(customer_json), orient="records")

    # 1. Türetilmiş kolonlar - Adapting for logistics data
    df_logistics["extracted_at"]    = pd.to_datetime(df_logistics["extracted_at"])
    df_logistics["shipping_month"]   = df_logistics["extracted_at"].dt.to_period("M").astype(str)
    df_logistics["shipping_year"]    = df_logistics["extracted_at"].dt.year

    # Convert EUR to USD for consistency if needed, assume 1 EUR = 1.08 USD for simulation
    df_logistics['shipping_cost_usd'] = df_logistics.apply(
        lambda row: row['shipping_cost'] * 1.08 if row['currency'] == 'EUR' else row['shipping_cost'],
        axis=1
    )

    # 2. Müşteri segmenti ile sol birleştirme (LEFT JOIN) - Skipping for now as customer_data is not extracted
    # df_enriched = df_sales.merge(
    #     df_customers[["customer_id", "segment"]],
    #     on="customer_id",
    #     how="left"
    # )
    # df_enriched["segment"] = df_enriched["segment"].fillna("Bilinmiyor")
    df_enriched = df_logistics # No enrichment for customer segments currently

    # 3. Bölge × Kategori bazlı özet - Adapting for logistics (origin-destination metrics)
    region_summary = (
        df_enriched.groupby(["origin", "destination"])
        .agg(
            total_shipments=("tracking_id", "count"),
            total_shipping_cost_usd=("shipping_cost_usd", "sum"),
            avg_weight=("weight_kg", "mean"),
        )
        .round(2)
        .reset_index()
    )

    # 4. Aylık trend - Adapting for logistics
    monthly_trend = (
        df_enriched.groupby("shipping_month")
        .agg(shipments=("tracking_id", "count"), total_cost_usd=("shipping_cost_usd", "sum"))
        .round(2)
        .reset_index()
    )

    ti.xcom_push("enriched_data",   df_enriched.to_json(orient="records", date_format="iso"))
    ti.xcom_push("region_summary",  region_summary.to_json(orient="records"))
    ti.xcom_push("monthly_trend",   monthly_trend.to_json(orient="records"))
    ti.xcom_push("total_revenue",   round(df_enriched["shipping_cost_usd"].sum(), 2)) # Using shipping cost as revenue equivalent

    ti.log_info(
        f"Zenginleştirme tamamlandı. "
        f"Toplam sevkiyat maliyeti: ${df_enriched['shipping_cost_usd'].sum():,.2f}"
    )


# =============================================================================
# ❹  LOAD — Hedef Sisteme Yazma Görevleri
# =============================================================================

def load_to_data_warehouse(ti: TaskInstance, db_path: str = DEFAULT_DB_PATH):
    """
    [LOAD TASK] Dönüştürülmüş veriyi Data Warehouse'a yaz.

    Gerçek senaryoda:
        - SQLAlchemy ile PostgreSQL/Redshift/Snowflake'e yazılır
        - df.to_sql("table", engine, if_exists="append") kullanılır
        - Büyük tablolar için COPY komutu veya bulk insert tercih edilir

    Burada SQLite ile gösterim yapılmaktadır.
    """
    ti.log_info(f"Data Warehouse'a yazılıyor: {db_path}")

    enriched_json = ti.xcom_pull("enrich_and_aggregate", "enriched_data")
    region_json   = ti.xcom_pull("enrich_and_aggregate", "region_summary")
    monthly_json  = ti.xcom_pull("enrich_and_aggregate", "monthly_trend")

    df_enriched      = pd.read_json(pd.io.common.StringIO(enriched_json), orient="records")
    df_region_summary = pd.read_json(pd.io.common.StringIO(region_json), orient="records")
    df_monthly       = pd.read_json(pd.io.common.StringIO(monthly_json), orient="records")

    # Hedef klasör yoksa oluştur (platform bağımsız).
    Path(db_path).parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)

    try:
        # fact_logistics → Ana lojistik gerçek tablosu (renamed from fact_sales)
        df_enriched.to_sql("fact_logistics", conn, if_exists="replace", index=False)
        ti.log_info(f"fact_logistics: {len(df_enriched)} satır yazıldı.")

        # dim_origin_destination → Bölge × kategori özet boyutu (renamed from dim_region_category)
        df_region_summary.to_sql(
            "dim_origin_destination", conn, if_exists="replace", index=False
        )
        ti.log_info(f"dim_origin_destination: {len(df_region_summary)} satır yazıldı.")

        # fact_monthly_trend → Aylık trend gerçek tablosu (kept same name, data adapted)
        df_monthly.to_sql("fact_monthly_trend", conn, if_exists="replace", index=False)
        ti.log_info(f"fact_monthly_trend: {len(df_monthly)} satır yazıldı.")

        # Yükleme meta verisi kaydet
        meta = pd.DataFrame([
            {
                "pipeline_run_id": hashlib.md5(
                    str(ti.execution_date).encode()
                ).hexdigest()[:8],
                "loaded_at"      : datetime.now().isoformat(),
                "row_count"      : len(df_enriched),
            }
        ])
        meta.to_sql("pipeline_runs", conn, if_exists="append", index=False)

        conn.commit()
        ti.xcom_push("db_path", db_path)
        ti.log_info("Tüm tablolar başarıyla yüklendi.")

    finally:
        conn.close()


# =============================================================================
# ❺  VALIDATE — Veri Kalite Kontrol Görevi
# =============================================================================

def run_data_quality_checks(ti: TaskInstance):
    """
    [VALIDATE TASK] Yüklenen veri üzerinde kalite kontrolleri çalıştır.

    Great Expectations veya Soda gibi araçlar bu aşamada kullanılır.
    Kontroller başarısız olursa pipeline başarısız sayılır ve
    downstream task'lar (bildirim hariç) atlanır.
    """
    ti.log_info("Veri kalite kontrolleri başlıyor...")

    db_path = ti.xcom_pull("load_to_data_warehouse", "db_path")
    conn    = sqlite3.connect(db_path)

    checks = []

    try:
        # ── Kontrol 1: Minimum kayıt sayısı ──────────────────────────────────
        # Changed table name from fact_sales to fact_logistics
        count = pd.read_sql("SELECT COUNT(*) AS cnt FROM fact_logistics", conn).iloc[0]["cnt"]
        checks.append({
            "name"   : "min_row_count",
            "passed" : count >= 100,
            "detail" : f"fact_logistics kayıt sayısı: {count} (min: 100)",
        })

        # ── Kontrol 2: Negatif net_amount yok mu? ────────────────────────────
        # Changed column name from net_amount to shipping_cost_usd
        neg_count = pd.read_sql(
            "SELECT COUNT(*) AS cnt FROM fact_logistics WHERE shipping_cost_usd < 0", conn
        ).iloc[0]["cnt"]
        checks.append({
            "name"   : "no_negative_amounts",
            "passed" : neg_count == 0,
            "detail" : f"Negatif maliyet sayısı: {neg_count} (beklenen: 0)",
        })

        # ── Kontrol 3: Null order_id yok mu? ─────────────────────────────────
        # Changed column name from order_id to tracking_id
        null_ids = pd.read_sql(
            "SELECT COUNT(*) AS cnt FROM fact_logistics WHERE tracking_id IS NULL", conn
        ).iloc[0]["cnt"]
        checks.append({
            "name"   : "no_null_tracking_ids",
            "passed" : null_ids == 0,
            "detail" : f"Null tracking_id sayısı: {null_ids} (beklenen: 0)",
        })

        # ── Kontrol 4: İskonto oranı mantıklı mı? ────────────────────────────
        # This check was for discount_pct in sales data. Adapting for weight_kg in logistics.
        bad_weight = pd.read_sql(
            "SELECT COUNT(*) AS cnt FROM fact_logistics "
            "WHERE weight_kg <= 0 OR weight_kg > 1000", conn
        ).iloc[0]["cnt"]
        checks.append({
            "name"   : "valid_weight_range",
            "passed" : bad_weight == 0,
            "detail" : f"Geçersiz ağırlık kayıtları: {bad_weight} (beklenen: 0)",
        })

        # ── Kontrol 5: Bölge çeşitliliği var mı? ────────────────────────────
        # Changed column from region to origin
        region_count = pd.read_sql(
            "SELECT COUNT(DISTINCT origin) AS cnt FROM fact_logistics", conn
        ).iloc[0]["cnt"]
        checks.append({
            "name"   : "origin_diversity",
            "passed" : region_count >= 3,
            "detail" : f"Farklı başlangıç bölgesi sayısı: {region_count} (min: 3)",
        })

    finally:
        conn.close()

    # Sonuçları raporla
    passed = sum(1 for c in checks if c["passed"])
    failed = len(checks) - passed

    ti.log_info(f"Kalite kontrol sonucu: {passed}/{len(checks)} geçti.")
    for c in checks:
        icon = "✓" if c["passed"] else "✗"
        ti.log_info(f"  {icon}  [{c['name']}] {c['detail']}")

    ti.xcom_push("quality_checks", checks)
    ti.xcom_push("quality_score",  round(passed / len(checks) * 100, 1))

    # Kritik kontrol başarısız → exception fırlat
    if failed > 0:
        failed_names = [c["name"] for c in checks if not c["passed"]]
        raise ValueError(f"Kalite kontrolü başarısız: {failed_names}")


# =============================================================================
# ❻  NOTIFY — Bildirim Görevi
# =============================================================================

def send_pipeline_notification(ti: TaskInstance, channel: str = "slack"):
    """
    [NOTIFY TASK] Pipeline tamamlandığında ekibi bilgilendir.

    Gerçek senaryoda:
        - Slack webhook: requests.post(SLACK_WEBHOOK_URL, json=payload)
        - E-posta: Airflow EmailOperator
        - PagerDuty: kritik hatalar için
        - Datadog/Grafana: metrik push
    """
    ti.log_info(f"Bildirim gönderiliyor ({channel})...")

    total_revenue = ti.xcom_pull("enrich_and_aggregate", "total_revenue")
    quality_score = ti.xcom_pull("run_data_quality_checks", "quality_score")
    removed_count = ti.xcom_pull("clean_and_validate_data", "removed_count")

    message = {
        "channel"      : f"#{channel}",
        "dag_id"       : ti.dag_id,
        "execution_date": ti.execution_date.strftime("%Y-%m-%d %H:%M"),
        "status"       : "✅ BAŞARILI",
        "metrics"      : {
            "toplam_maliyet"         : f"${total_revenue:,.2f}" if total_revenue else "N/A", # Changed from ciro to maliyet
            "kalite_skoru"        : f"%{quality_score}" if quality_score else "N/A",
            "temizlenen_kayitlar" : removed_count or 0,
        },
    }

    # Gerçekte: requests.post(webhook_url, json=message)
    ti.log_info(f"Bildirim içeriği: {json.dumps(message, ensure_ascii=False, indent=2)}")
    ti.log_info("Bildirim başarıyla gönderildi.")


# =============================================================================
# ❼  DAG TANIMLAMASI — Pipeline'ı Oluştur ve Bağla
# =============================================================================

def create_etl_dag() -> DAG:
    """
    ETL pipeline DAG'ını oluştur.

    Task bağımlılık grafiği:

        extract_sales ──┐
                        ├──► clean_validate ──► enrich_aggregate ──► load_dw ──► quality_check ──► notify
        extract_customers ──┘

    Gerçek Airflow'da bu fonksiyon modül düzeyinde çağrılır ve
    scheduler otomatik olarak DAG'ı keşfeder.
    """

    # Varsayılan task argümanları (tüm task'lara uygulanır)
    default_args = {
        "owner"           : "data-team",
        "depends_on_past" : False,         # Önceki çalışmanın bitmesini bekleme
        "email_on_failure": True,
        "email_on_retry"  : False,
    }

    dag = DAG(
        dag_id           ="etl_logistics_pipeline", # Renamed dag_id
        description      ="Lojistik verisi ETL pipeline'ı — günlük çalışır", # Updated description
        schedule_interval="0 3 * * *",     # Her gün 03:00'da çalış
        start_date       =datetime(2024, 1, 1),
        catchup          =False,           # Geçmiş çalışmaları telafi etme
        tags             =["etl", "lojistik", "gunluk"], # Updated tags
        default_args     =default_args,
    )

    # ── Task Tanımlamaları ─────────────────────────────────────────────────
    # Removed t_extract_sales and t_extract_customers as they are not defined.
    # Using extract_logistics_data and extract_warehouse_stock from the simulation.

    t_extract_logistics = PythonOperator(
        task_id          ="extract_logistics_data",
        python_callable  =extract_logistics_data,
        op_kwargs        ={"source_system": "SAP_ERP"}, # Adjusted kwargs
        retries          =2,
        group            ="extract",
    )

    t_extract_warehouse_stock = PythonOperator(
        task_id         ="extract_warehouse_stock",
        python_callable =extract_warehouse_stock,
        retries         =2,
        group           ="extract",
    )

    t_clean = PythonOperator(
        task_id         ="clean_and_validate_data",
        python_callable =clean_and_validate_data,
        retries         =1,
        group           ="transform",
    )

    t_enrich = PythonOperator(
        task_id         ="enrich_and_aggregate",
        python_callable =enrich_and_aggregate,
        retries         =1,
        group           ="transform",
    )

    t_load = PythonOperator(
        task_id         ="load_to_data_warehouse",
        python_callable =load_to_data_warehouse,
        op_kwargs       ={"db_path": DEFAULT_DB_PATH},
        retries         =3,
        group           ="load",
    )

    t_quality = PythonOperator(
        task_id         ="run_data_quality_checks",
        python_callable =run_data_quality_checks,
        retries         =0,
        group           ="validate",
    )

    t_notify = PythonOperator(
        task_id         ="send_pipeline_notification",
        python_callable =send_pipeline_notification,
        op_kwargs       ={"channel": "data-alerts"},
        retries         =2,
        group           ="notify",
    )

    # ── Bağımlılıkları Tanımla (>> operatörü veya set_upstream) ───────────
    #
    # Airflow'da iki eşdeğer yazım vardır:
    #   Yöntem 1 (>> operatörü):   t_extract >> t_clean >> t_enrich
    #   Yöntem 2 (set_upstream):   t_clean.set_upstream(t_extract)
    #
    # Paralel extract task'ları temizleme öncesi birleşir:
    # Adjusted dependencies for the new extract tasks
    t_extract_logistics >> t_clean
    t_extract_warehouse_stock >> t_clean # Assuming warehouse stock also feeds into clean_and_validate_data
    t_clean >> t_enrich >> t_load >> t_quality >> t_notify

    # Tüm task'ları DAG'a kaydet
    for task in [
        t_extract_logistics, t_extract_warehouse_stock,
        t_clean, t_enrich, t_load, t_quality, t_notify,
    ]:
        dag.add_task(task)

    return dag


# =============================================================================
# ❽  ANA GİRİŞ NOKTASI
# =============================================================================

if __name__ == "__main__":
    """
    Bu blok sadece doğrudan `python etl_airflow_pipeline.py` ile
    çalıştırıldığında tetiklenir.

    Gerçek Airflow'da DAG dosyaları modül olarak import edilir;
    __main__ bloğuna gerek yoktur.

    Yerel geliştirme & test için bu simülasyon kullanışlıdır.
    """

    print("\n" + "█" * 70)
    print("  ETL / ELT Pipeline — Apache Airflow Mimarisi")
    print("  Lojistik Veri Ambarı Güncelleme Pipeline'ı") # Updated description
    print("█" * 70)

    # DAG'ı oluştur
    satis_dag = create_etl_dag()

    # Manuel tetikleme (gerçekte: airflow dags trigger etl_satis_pipeline)
    results = satis_dag.run(execution_date=datetime.now())

    # Detaylı sonuç raporu
    print("\n📋  DETAYLI TASK SONUÇLARI")
    print("─" * 55)
    for task_id, ti in results.items():
        state_icon = {
            TaskState.SUCCESS: "✅",
            TaskState.FAILED : "❌",
            TaskState.SKIPPED: "⏭ ",
        }.get(ti.state, "❓")
        print(
            f"  {state_icon}  {task_id:<35}  "
            f"{ti.duration:.2f}s  [{ti.state.value}]"
        )

    # XCom özetini göster
    xcom_store = next(iter(results.values())).xcom
    print("\n🗂   XCOM DEĞERLERİ (Task'lar Arası Veri)")
    print("─" * 55)
    for key, value in xcom_store._store.items():
        display = value if not isinstance(value, str) or len(value) < 60 else value[:57] + "..."
        print(f"  {key:<45} → {display}")

    # Veritabanı özeti
    db_path = DEFAULT_DB_PATH
    if Path(db_path).exists():
        print(f"\n💾  VERİ AMBARI ÖZETI ({db_path})")
        print("─" * 55)
        conn = None # Initialize conn to None
        try:
            conn = sqlite3.connect(db_path)
            # Changed table names
            for table in ["fact_logistics", "dim_origin_destination", "fact_monthly_trend", "pipeline_runs"]:
                try:
                    count = pd.read_sql(f"SELECT COUNT(*) AS c FROM {table}", conn).iloc[0]["c"]
                    print(f"  Tablo: {table:<30} {count:>6} satır")
                except Exception:
                    pass
        finally:
            if conn: # Only close if connection was successfully opened
                conn.close()

    print("\n✨  Pipeline simülasyonu tamamlandı.\n")


██████████████████████████████████████████████████████████████████████
  ETL / ELT Pipeline — Apache Airflow Mimarisi
  Lojistik Veri Ambarı Güncelleme Pipeline'ı
██████████████████████████████████████████████████████████████████████

══════════════════════════════════════════════════════════════════════
  🚀  DAG ÇALIŞIYOR: etl_logistics_pipeline
  📅  Execution Date: 2026-07-08 06:58:14
  📋  Açıklama: Lojistik verisi ETL pipeline'ı — günlük çalışır
══════════════════════════════════════════════════════════════════════

  📥  Task: extract_logistics_data  [EXTRACT]
      ──────────────────────────────────────────────────
      ✓  Süre: 0.03s

  📥  Task: extract_warehouse_stock  [EXTRACT]
      ──────────────────────────────────────────────────
      ✓  Süre: 0.00s

  ⚙️  Task: clean_and_validate_data  [TRANSFORM]
      ──────────────────────────────────────────────────
      ✓  Süre: 0.04s

  ⚙️  Task: enrich_and_aggregate  [TRANSFORM]
      ─────────────────────────────────────────────